# RISC-V core: Verilog to chip layout (SkyWater 130nm) in Colab

Takes the 5-stage RV32I core from
[riscv-pipeline-rv32i](https://github.com/Raushan-kumar0/riscv-pipeline-rv32i)
through the full open-source chip flow (LibreLane): synthesis → floorplan →
placement → clock tree → routing → signoff (timing, DRC, LVS) → GDSII layout.

**How to use:** *Runtime → Run all*, then keep this tab open. Setup takes roughly
10–20 minutes, and the flow itself about an hour on a free Colab machine.

The two setup cells below are taken from LibreLane's official Colab notebook
(github.com/librelane/librelane, Apache-2.0).

In [ ]:
# @title Setup Nix {display-mode: "form"}
# @markdown <img src="https://raw.githubusercontent.com/NixOS/nixos-artwork/51a27e4a011e95cb559e37d32c44cf89b50f5154/logo/nix-snowflake-colours.svg" width="32"/>
# @markdown
# @markdown Nix is a package manager with an emphasis on reproducible builds,
# @markdown and it is the primary method for installing LibreLane.
# @markdown
# @markdown This step installs the Nix package manager and enables the
# @markdown FOSSi Foundation Nix Cache.
# @markdown
# @markdown If you're not in a Colab, this just sets the environment variables.
# @markdown You will need to install Nix and enable flakes on your own following
# @markdown [this guide](https://librelane.readthedocs.io/en/stable/getting_started/common/nix_installation/index.html).
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

In [ ]:
# @title Get LibreLane {display-mode: "form"}
# @markdown Click the ▷ button to download and install LibreLane.
# @markdown
# @markdown This will install LibreLane's tool dependencies using Nix,
# @markdown and LibreLane itself using PIP.
# @markdown
# @markdown Note that `python3-tk` may need to be installed using your OS's
# @markdown package manager.
import os
import yaml
import subprocess
import IPython

librelane_version = "latest"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

### 1. Get the processor's Verilog (only the core; memories stay outside the chip)

In [ ]:
import os, shutil
os.chdir("/content")
!rm -rf riscv-pipeline-rv32i rtl2gds
!git clone -q https://github.com/Raushan-kumar0/riscv-pipeline-rv32i.git
os.makedirs("rtl2gds/src", exist_ok=True)
for f in ["riscv_defs.vh", "alu.v", "regfile.v", "control.v", "imm_gen.v", "branch_cmp.v", "riscv_core.v"]:
    shutil.copy(f"riscv-pipeline-rv32i/rtl/{f}", "rtl2gds/src/")
print(sorted(os.listdir("rtl2gds/src")))

### 2. Flow settings and timing rules
`CLOCK_PERIOD` is in nanoseconds (25 ns = 40 MHz). For the timing-closure
exercise, lower it later and re-run steps 2–5.

In [ ]:
%%writefile /content/rtl2gds/config.json
{
  "DESIGN_NAME": "riscv_core",
  "VERILOG_FILES": "dir::src/*.v",
  "VERILOG_INCLUDE_DIRS": "dir::src",
  "CLOCK_PORT": "clk",
  "CLOCK_PERIOD": 25,
  "PNR_SDC_FILE": "dir::constraints.sdc",
  "SIGNOFF_SDC_FILE": "dir::constraints.sdc",
  "FP_CORE_UTIL": 40,
  "PL_TARGET_DENSITY_PCT": "expr::($FP_CORE_UTIL + 10.0)"
}

In [ ]:
%%writefile /content/rtl2gds/constraints.sdc
# constraints.sdc - the timing rules the layout tools must meet.
# An SDC file answers three questions: how fast is the clock, how much of each
# cycle does the outside world use up, and what do the pins connect to?

# 1. The clock: one clock on port 'clk'. Its period (ns) comes from CLOCK_PERIOD in config.json.
create_clock [get_ports clk] -name clk -period $::env(CLOCK_PERIOD)

# Real clocks jitter and arrive with skew; reserve a safety margin for that.
set_clock_uncertainty 0.25 [get_clocks clk]
set_clock_transition  0.15 [get_clocks clk]

# 2. I/O budget: assume the memories outside the core use 20% of each cycle
#    on the way in and 20% on the way out. The core gets the rest.
set io_delay [expr {$::env(CLOCK_PERIOD) * 0.2}]
set clk_port   [get_ports clk]
set clk_index  [lsearch [all_inputs] $clk_port]
set data_inputs [lreplace [all_inputs] $clk_index $clk_index]
set_input_delay  $io_delay -clock [get_clocks clk] $data_inputs
set_output_delay $io_delay -clock [get_clocks clk] [all_outputs]

# 3. What the pins connect to: inputs are driven by a small inverter,
#    outputs drive about 33 fF (0.033 pF) of load.
set_driving_cell -lib_cell sky130_fd_sc_hd__inv_2 -pin Y $data_inputs
set_load 0.033 [all_outputs]

# Keep any one gate from driving too many others (long, slow nets).
set_max_fanout 10 [current_design]

### 3. Run the whole flow (the long step)
Before starting: **plug in the laptop and stop it from sleeping**
(Windows: Settings → System → Power → set *sleep* to *Never* while plugged in), and keep this tab open.
Each stage prints a `Running '...'` line as it starts.

In [ ]:
# Run the flow INSIDE this notebook (like LibreLane's official Colab example).
# LibreLane switches on its built-in Colab fixes only when it runs inside the notebook itself.
import os
os.environ.pop("PYTHONPATH", None)          # undo any earlier workaround
os.chdir("/content/rtl2gds")
from librelane.flows import Flow
Classic = Flow.factory.get("Classic")
flow = Classic("/content/rtl2gds/config.json", design_dir="/content/rtl2gds",
               pdk="sky130A", pdk_root=os.path.expanduser("~/.ciel"))
try:
    flow.start()
    print("\n==== FLOW FINISHED ====")
except Exception as e:
    print("\n==== FLOW STOPPED ====")
    print(e)

**If the page disconnects** while the flow is running, reconnect and run **only** the cell below:
it shows how far the flow got and the latest log lines.

In [ ]:
import os, glob
runs = sorted(glob.glob("/content/rtl2gds/runs/*"), key=os.path.getmtime)
if not runs:
    print("No run folder: the Colab machine was reset. Use Runtime -> Run all again.")
else:
    steps = sorted(d for d in glob.glob(os.path.join(runs[-1], "*")) if os.path.isdir(d) and os.path.basename(d)[:1].isdigit())
    print("Run:", os.path.basename(runs[-1]), "| steps started:", len(steps), "| latest:", os.path.basename(steps[-1]) if steps else "-")
    log = os.path.join(runs[-1], "flow.log")
    if os.path.exists(log):
        print("---- last 20 lines of flow.log ----")
        print("".join(open(log).readlines()[-20:]))

### 4. Results

In [ ]:
import json, glob, os
runs = sorted(glob.glob("/content/rtl2gds/runs/*"), key=os.path.getmtime)
assert runs, "No run folder found: did step 3 start?"
run = runs[-1]
final_metrics = os.path.join(run, "final", "metrics.json")
if os.path.exists(final_metrics):
    m = json.load(open(final_metrics))
    print("Run:", os.path.basename(run), "(finished)")
else:
    states = sorted(glob.glob(os.path.join(run, "*", "state_out.json")))
    m = json.load(open(states[-1]))["metrics"] if states else {}
    print("Run:", os.path.basename(run), "- flow did NOT finish; showing metrics from the last completed step")
    done = [os.path.basename(os.path.dirname(p)) for p in states]
    print("Steps that finished:", len(done), "| last one:", done[-1] if done else "none")
    log = os.path.join(run, "flow.log")
    if os.path.exists(log):
        print("---- last 15 lines of flow.log ----")
        print("".join(open(log).readlines()[-15:]))

def pick(prefix, contains=None):
    return {k: v for k, v in m.items() if k.startswith(prefix) and (contains is None or contains in k)}

# clock period of THIS run (from the run's own saved settings, not the current config file)
resolved = os.path.join(run, "resolved.json")
period = json.load(open(resolved if os.path.exists(resolved) else "/content/rtl2gds/config.json"))["CLOCK_PERIOD"]
rows = [
    ("Clock period (ns)", period),
    ("Standard cells", m.get("design__instance__count__stdcell", m.get("design__instance__count"))),
    ("Cell area (um^2)", m.get("design__instance__area__stdcell", m.get("design__instance__area"))),
    ("Core area (um^2)", m.get("design__core__area")),
    ("Die area (um^2)", m.get("design__die__area")),
    ("Worst setup slack, all corners (ns)", m.get("timing__setup__ws")),
    ("Worst hold slack, all corners (ns)", m.get("timing__hold__ws")),
    ("Setup violations (count)", m.get("timing__setup_vio__count")),
    ("Hold violations (count)", m.get("timing__hold_vio__count")),
    ("Routing DRC errors", m.get("route__drc_errors")),
    ("Magic DRC errors", m.get("magic__drc_error__count")),
    ("LVS errors", m.get("design__lvs_error__count")),
    ("Total power (W)", m.get("power__total")),
]
for name, val in rows:
    print(f"{name:40s} {val}")

print("\nSetup slack per corner (ns):")
for k, v in sorted(pick("timing__setup__ws__corner").items()):
    print(f"   {k.split(':', 1)[-1]:28s} {v}")
tt = [v for k, v in pick("timing__setup__ws__corner", "tt").items() if isinstance(v, (int, float))]
if tt:
    ws = min(tt)
    print(f"\nTypical corner: worst slack {ws:.3f} ns -> estimated max clock ~ {1000.0 / (period - ws):.1f} MHz")

### 5. The finished chip layout

In [ ]:
from IPython.display import Image, display
pngs = sorted(glob.glob(os.path.join(run, "*klayout-render*", "*.png")))
if pngs:
    display(Image(pngs[-1], width=800))
    print(pngs[-1])
else:
    print("No layout picture yet (the render step runs near the end of the flow).")

### 6. Download the results for GitHub
Downloads a small zip with the layout picture, `metrics.json` and the timing
summaries. (The GDS file itself is too big for GitHub's web upload, so it stays here.)

In [ ]:
import zipfile
from google.colab import files
out = "/content/rtl2gds_results.zip"
with zipfile.ZipFile(out, "w") as z:
    for p in pngs[-1:]:
        z.write(p, "layout.png")
    if os.path.exists(final_metrics):
        z.write(final_metrics, "metrics.json")
    for i, rpt in enumerate(sorted(glob.glob(os.path.join(run, "*sta*", "summary.rpt")))):
        z.write(rpt, f"timing/{i:02d}_{os.path.basename(os.path.dirname(rpt))}_summary.rpt")
    z.write("/content/rtl2gds/config.json", "config.json")
    for p in glob.glob(os.path.join(run, "*.log")):
        z.write(p, "logs/" + os.path.basename(p))
    z.write("/content/rtl2gds/constraints.sdc", "constraints.sdc")
print("zip contents:", zipfile.ZipFile(out).namelist())
files.download(out)